# Station Stacking v18.1 NBM Wunderground Ablation - KATL

Experimental notebook for `KATL`.

This v18.1 NBM ablation keeps the v11 remaining-warmup ridge-stack backbone, requires Wunderground station-history labels, and adds only the coverage-gated NBM hourly-curve shard features. It writes isolated artifacts to `data/calibration/station_stacking_v18_1_nbm`.


In [1]:
from pathlib import Path
import os
import sys
import warnings

warnings.filterwarnings("ignore", message="IProgress not found.*")
warnings.filterwarnings("ignore", message="Skipping features without any observed values.*")

PROJECT_ROOT = Path.cwd().resolve()
while not (PROJECT_ROOT / "src" / "calibration" / "station_stacking.py").exists():
    if PROJECT_ROOT.parent == PROJECT_ROOT:
        raise RuntimeError("Could not find project root containing src/calibration/station_stacking.py")
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

os.environ["WEATHER_RESEARCH_INCLUDE_DIRECT_NBM"] = "1"

STATION_ID = "KATL"
PROVIDERS = ("gfs", "hrrr", "nbm")
TREND_COLUMNS = [
    "observed_temp_change_last_1h_f",
    "observed_temp_change_last_3h_f",
    "observed_morning_warmup_rate_f_per_hour",
    "observed_high_so_far_change_since_9am_f",
]
TIMING_MODE = "same_day_11am_live_safe"
FAST_MODE = False
OPTUNA_TRIALS = 100
STACK_OPTUNA_TRIALS = 100
OPTUNA_STARTUP_TRIALS = 40
STACK_OPTUNA_STARTUP_TRIALS = 40
OPTUNA_METRIC = "mae_f"
OPTUNA_VERBOSE = True
MODEL_VERSION = "station_high_regressor_v18_1_nbm_settlement_stack"
PROJECT_ROOT


WindowsPath('D:/dev/weather-research')

In [2]:
import numpy as np
import pandas as pd

from src.export_station_stacking_v2_models import export_station_model_weights
from src.calibration.station_stacking import (
    StationStackingConfig,
    V11_DROPPED_FEATURE_COLUMNS,
    V11_FEATURE_COLUMNS,
    YEAR_SPLIT_EXPANDING_FOLDS,
    missing_model_dependencies,
    provider_availability,
    run_station_year_split_experiment,
)


## V11 Contract

`feature_version="v18_1_nbm"` adds only the coverage-gated NBM hourly-curve shard features while selecting by validation MAE.


In [3]:
fold_spec = pd.DataFrame(
    [
        {
            "fold": fold.name,
            "train_start_year": fold.train_start_year,
            "train_end_year": fold.train_end_year,
            "validation_year": fold.validation_year,
        }
        for fold in YEAR_SPLIT_EXPANDING_FOLDS
    ]
)

fold_spec


,fold,train_start_year,train_end_year,validation_year
0,fold_2021_2023_to_2024,2021,2023,2024
1,fold_2021_2024_to_2025,2021,2024,2025


In [4]:
V11_FEATURE_COLUMNS, sorted(V11_DROPPED_FEATURE_COLUMNS)


(['v2_recent_heat_anomaly_f',
  'v2_recent_heat_momentum_f',
  'v2_morning_warmup_to_consensus_f',
  'v2_consensus_minus_7d_actual_f',
  'v2_spread_per_warmup_f',
  'v2_humidity_warmup_interaction',
  'v3_high_so_far_above_current_f',
  'v3_remaining_warmup_from_high_so_far_f',
  'v3_high_so_far_minus_lag_1d_f',
  'v3_high_so_far_minus_7d_actual_f',
  'v3_remaining_warmup_per_spread_f',
  'v3_humidity_remaining_warmup_interaction',
  'v4_forecast_precip_total_mean_mm',
  'v4_forecast_precip_total_max_mm',
  'v4_forecast_precip_total_spread_mm',
  'v4_forecast_precip_max_1h_mean_mm',
  'v4_forecast_precip_hours_mean',
  'v4_forecast_precip_intensity_mean',
  'v4_forecast_precip_intensity_max',
  'v4_any_forecast_precip',
  'v4_all_forecast_precip',
  'v4_observed_precip_any',
  'v4_observed_precip_recent_mm_est',
  'v4_forecast_total_minus_observed_recent_mm',
  'v4_forecast_observed_precip_match',
  'v4_forecast_wet_observed_dry',
  'v4_observed_wet_forecast_dry',
  'v4_precip_humidity

## Data Availability


In [5]:
availability = provider_availability(
    PROJECT_ROOT,
    timing_mode=TIMING_MODE,
    providers=PROVIDERS,
)

availability.loc[availability["station_id"].eq(STATION_ID)]


,station_id,provider,row_count,first_contract_date,last_contract_date
0,KATL,gfs,1998,2021-01-01,2026-06-21
1,KATL,hrrr,1998,2021-01-01,2026-06-21
2,KATL,nbm,1997,2021-01-01,2026-06-21


## Model Scores


In [6]:
missing_packages = missing_model_dependencies()
if missing_packages:
    raise ImportError(
        "Missing station-stacking ML packages: "
        + ", ".join(missing_packages)
        + ". Install them with: python -m pip install -r requirements.txt"
    )

config = StationStackingConfig(
    station_id=STATION_ID,
    project_root=PROJECT_ROOT,
    timing_mode=TIMING_MODE,
    providers=PROVIDERS,
    fast_mode=FAST_MODE,
    optuna_trials=OPTUNA_TRIALS,
    stack_optuna_trials=STACK_OPTUNA_TRIALS,
    optuna_startup_trials=OPTUNA_STARTUP_TRIALS,
    stack_optuna_startup_trials=STACK_OPTUNA_STARTUP_TRIALS,
    optuna_metric=OPTUNA_METRIC,
    optuna_verbose=OPTUNA_VERBOSE,
    feature_version="v18_1_nbm",
    target_mode="remaining_warmup",
    target_source="wunderground_only",
    base_model_methods=("xgboost", "lightgbm", "catboost"),
    stack_enabled=True,
    hyperparameter_space="wide_plus",
    year_split_folds=YEAR_SPLIT_EXPANDING_FOLDS,
    year_split_test_train_years=(2021, 2025),
    year_split_test_year=2026,
    output_dir=PROJECT_ROOT / "data" / "calibration" / "station_stacking_v18_1_nbm",
)

config.resolved_optuna_storage_path()


WindowsPath('D:/dev/weather-research/data/calibration/station_stacking_v18_1_nbm/KATL_optuna.sqlite3')

In [7]:
result = run_station_year_split_experiment(config)
result.scoreboard


[I 2026-07-10 06:09:42,371] A new study created in RDB with name: KATL_v18_1_nbm_remaining_warmup_base_xgboost_mae_f_wide_plus
[I 2026-07-10 06:09:55,508] Trial 0 finished with value: 1.65511499241476 and parameters: {'n_estimators': 2278, 'learning_rate': 0.18404304074388048, 'max_depth': 9, 'min_child_weight': 2.481040974867813, 'gamma': 2.340279606636548, 'subsample': 0.4513964382185317, 'colsample_bytree': 0.3877543479093296, 'reg_alpha': 2.478071022662141, 'reg_lambda': 0.6132587025321562}. Best is trial 0 with value: 1.65511499241476.
[I 2026-07-10 06:10:14,658] Trial 1 finished with value: 1.4093953388535114 and parameters: {'n_estimators': 4263, 'learning_rate': 0.0005682336350005583, 'max_depth': 12, 'min_child_weight': 21.368329072358772, 'gamma': 3.185086660174142, 'subsample': 0.4681862286846154, 'colsample_bytree': 0.46921293140473197, 'reg_alpha': 4.476173538513514e-07, 'reg_lambda': 0.20253776634919213}. Best is trial 1 with value: 1.4093953388535114.
[I 2026-07-10 06:10

,period,method,count,mae_f,rmse_f
0,validation_2024_2025,xgboost,729,1.245220,1.677117
1,validation_2024_2025,lightgbm,729,1.314057,1.750095
2,validation_2024_2025,catboost,729,1.288386,1.717139
3,validation_2024_2025,provider_mean,729,2.102767,3.049105
4,validation_2024_2025,provider_median,729,2.022252,2.951401
5,validation_2024_2025,nbm_raw,729,1.987547,2.848066
6,validation_2024_2025,hrrr_raw,729,2.790393,3.978949
7,validation_2024_2025,gfs_raw,729,2.859717,3.849158
8,test_2026,xgboost,171,1.342726,1.917886
9,test_2026,lightgbm,171,1.429659,2.020070


In [8]:
exported_weights = export_station_model_weights(
    project_root=PROJECT_ROOT,
    station_id=STATION_ID,
    artifact_dir=config.resolved_output_dir(),
    model_version=MODEL_VERSION,
    timing_mode=config.timing_mode,
    providers=tuple(config.providers),
    feature_version=config.effective_feature_version,
    optuna_metric=config.effective_optuna_metric,
    target_mode=config.effective_target_mode,
    target_source=config.effective_target_source,
    base_model_methods=tuple(config.effective_base_model_methods),
    stack_enabled=config.stack_enabled,
    source_pipeline="notebooks/station_stacking_v18_1",
)

exported_weights.bundle_path, exported_weights.manifest_path


(WindowsPath('D:/dev/weather-research/data/calibration/station_stacking_v18_1_nbm/model_weights/KATL_station_high_regressor_v18_1_nbm_settlement_stack.joblib'),
 WindowsPath('D:/dev/weather-research/data/calibration/station_stacking_v18_1_nbm/model_weights/KATL_station_high_regressor_v18_1_nbm_settlement_stack.json'))

## V11 Feature Coverage


In [9]:
v11_feature_coverage = (
    result.features[V11_FEATURE_COLUMNS]
    .notna()
    .mean()
    .mul(100)
    .sort_values(ascending=False)
    .rename("coverage_pct")
    .reset_index()
    .rename(columns={"index": "feature"})
)

v11_feature_coverage


,feature,coverage_pct
0,v2_spread_per_warmup_f,100.000000
1,v2_morning_warmup_to_consensus_f,100.000000
2,v3_remaining_warmup_from_high_so_far_f,100.000000
3,v3_high_so_far_above_current_f,100.000000
4,v2_humidity_warmup_interaction,100.000000
5,v4_observed_precip_recent_mm_est,100.000000
6,v4_forecast_wet_observed_dry,100.000000
7,v4_forecast_observed_precip_match,100.000000
8,v3_humidity_remaining_warmup_interaction,100.000000
9,v3_remaining_warmup_per_spread_f,100.000000


In [10]:
result.feature_columns.loc[result.feature_columns["feature"].isin(V11_FEATURE_COLUMNS)]


,feature,kind
3,observed_temp_change_last_1h_f,numeric
4,observed_temp_change_last_3h_f,numeric
5,observed_morning_warmup_rate_f_per_hour,numeric
6,observed_high_so_far_change_since_9am_f,numeric
20,v2_recent_heat_anomaly_f,numeric
21,v2_recent_heat_momentum_f,numeric
22,v2_morning_warmup_to_consensus_f,numeric
23,v2_consensus_minus_7d_actual_f,numeric
24,v2_spread_per_warmup_f,numeric
25,v2_humidity_warmup_interaction,numeric


## Dropped Feature Check


In [11]:
dropped_present = result.feature_columns.loc[
    result.feature_columns["feature"].isin(V11_DROPPED_FEATURE_COLUMNS)
]

dropped_present


,feature,kind


## Morning Trend Coverage


In [12]:
trend_coverage = (
    result.features[TREND_COLUMNS]
    .notna()
    .mean()
    .mul(100)
    .sort_values(ascending=False)
    .rename("coverage_pct")
    .reset_index()
    .rename(columns={"index": "feature"})
)

trend_coverage


,feature,coverage_pct
0,observed_temp_change_last_1h_f,99.8999
1,observed_temp_change_last_3h_f,99.8999
2,observed_morning_warmup_rate_f_per_hour,99.8999
3,observed_high_so_far_change_since_9am_f,99.8999


## Rounded Within 1F Accuracy


In [13]:
preds = pd.concat(
    [
        result.validation_predictions.assign(period="validation_2024_2025"),
        result.test_predictions.assign(period="oof_2026"),
    ],
    ignore_index=True,
)

predicted_high = pd.to_numeric(preds["predicted_high_f"], errors="coerce")
preds["predicted_high_rounded_f"] = np.floor(predicted_high + 0.5)
preds["within_1f_after_round"] = (
    pd.to_numeric(preds["actual_high_f"], errors="coerce") - preds["predicted_high_rounded_f"]
).abs().le(1)

within_1f_accuracy_by_period = (
    preds
    .dropna(subset=["actual_high_f", "predicted_high_rounded_f"])
    .groupby(["period", "method"], as_index=False)
    .agg(
        count=("within_1f_after_round", "size"),
        within_1f_count=("within_1f_after_round", "sum"),
        within_1f_accuracy_pct=("within_1f_after_round", lambda x: x.mean() * 100),
    )
    .sort_values(["period", "within_1f_accuracy_pct"], ascending=[True, False])
)

within_1f_accuracy_by_period


,period,method,count,within_1f_count,within_1f_accuracy_pct
11,oof_2026,xgboost,171,121,70.760234
0,oof_2026,catboost,171,118,69.005848
10,oof_2026,ridge_stack,171,117,68.421053
4,oof_2026,guarded_blend_cap_3f,171,111,64.912281
3,oof_2026,guarded_blend_cap_2f,171,110,64.327485
6,oof_2026,lightgbm,171,106,61.988304
2,oof_2026,guarded_blend_cap_1f,171,104,60.818713
7,oof_2026,nbm_raw,171,91,53.216374
9,oof_2026,provider_median,171,84,49.122807
8,oof_2026,provider_mean,171,78,45.614035


## Version Comparison


In [14]:
comparison_frames = []
for version, folder in [
    ("v1", PROJECT_ROOT / "data" / "calibration" / "station_stacking"),
    ("v2", PROJECT_ROOT / "data" / "calibration" / "station_stacking_v2"),
    ("v3", PROJECT_ROOT / "data" / "calibration" / "station_stacking_v3"),
    ("v4", PROJECT_ROOT / "data" / "calibration" / "station_stacking_v4"),
    ("v5", PROJECT_ROOT / "data" / "calibration" / "station_stacking_v5"),
    ("v6", PROJECT_ROOT / "data" / "calibration" / "station_stacking_v6"),
    ("v7", PROJECT_ROOT / "data" / "calibration" / "station_stacking_v7"),
    ("v8", PROJECT_ROOT / "data" / "calibration" / "station_stacking_v8"),
    ("v9", PROJECT_ROOT / "data" / "calibration" / "station_stacking_v9"),
    ("v10", PROJECT_ROOT / "data" / "calibration" / "station_stacking_v10"),
    ("v11", PROJECT_ROOT / "data" / "calibration" / "station_stacking_v11"),
]:
    path = folder / f"{STATION_ID}_year_split_scoreboard.csv"
    if path.exists():
        frame = pd.read_csv(path)
        frame["version"] = version
        comparison_frames.append(frame)

version_comparison = pd.concat(comparison_frames, ignore_index=True) if comparison_frames else pd.DataFrame()
if not version_comparison.empty:
    version_comparison = version_comparison.sort_values(["period", "mae_f", "version", "method"]).reset_index(drop=True)
version_comparison


,period,method,count,mae_f,rmse_f,version
0,test_2026,lightgbm,137,1.529121,2.224566,v11
1,test_2026,ridge_stack,137,1.531900,2.209782,v11
2,test_2026,lightgbm,125,1.547989,2.093915,v5
3,test_2026,ridge_stack,125,1.566708,2.118374,v5
4,test_2026,xgboost,125,1.569277,2.117086,v5
...,...,...,...,...,...,...
111,validation_2024_2025,gfs_raw,541,3.557627,5.060559,v1
112,validation_2024_2025,hrrr_raw,664,3.630969,5.083107,v2
113,validation_2024_2025,hrrr_raw,664,3.630969,5.083107,v3
114,validation_2024_2025,hrrr_raw,664,3.630969,5.083107,v4


## 2026 OOF Weather Brackets


In [15]:
result.bracket_metrics


,method,count,mae_f,rmse_f,bucket_log_loss,bracket_accuracy_pct,p95_absolute_error_f,large_miss_5f_pct
0,xgboost,171,1.342726,1.917886,1.425946,45.02924,3.706235,2.339181
1,lightgbm,171,1.429659,2.020070,1.446169,44.444444,3.892889,1.754386
2,catboost,171,1.463412,2.085443,1.511357,43.274854,4.224616,2.923977
3,ridge_stack,171,1.371293,1.928701,1.433301,45.614035,3.975188,1.754386
4,guarded_blend_cap_1f,171,1.829174,2.995393,1.821773,36.25731,4.578531,4.678363
5,guarded_blend_cap_2f,171,1.692701,2.817678,1.783725,40.935673,4.470859,4.093567
6,guarded_blend_cap_3f,171,1.642169,2.697265,1.746556,41.520468,4.463171,4.093567
7,provider_mean,171,2.145562,3.337353,1.877072,34.502924,5.357125,6.432749
8,provider_median,171,2.126776,3.276249,1.874809,33.918129,5.905997,7.017544
9,nbm_raw,171,2.088445,3.229287,1.869976,35.672515,5.905997,7.602339
